# Rozmiary wykrytych twarzy

Czyta gotowe pliki pomiaru `face_sizes_*.json` z `results/measurements/` i układa z nich tabele rozkładu rozmiarów twarzy oraz po jednym rysunku na część materiału. Sam niczego nie mierzy.

Pomiar powstaje przy obniżonej podłodze detekcji (`--floor 8`), bo produkcyjny bufor twarzy nie trzyma niczego poniżej progu `MIN_FACE_PX` i odsetka odrzuconych nie da się z niego odczytać.

**Wymaga:** pomiaru obu części (część testowa jest tu wyjątkiem od bramki - rysunek opisowy, bez Recall; `docs/05_instrukcja_test.md`):

```powershell
python scripts/face_size_survey.py --split dev
python scripts/face_size_survey.py --split test
```

"Krótszy bok" w tabelach to mniejszy z dwóch boków ramki twarzy w pikselach oryginalnej klatki - ta wielkość decyduje o tym, ile informacji zostaje po przeskalowaniu wycinka do wejścia ArcFace'a. Udziały są w procentach, separatorem dziesiętnym jest przecinek, wiersz bez pomiaru wypisuje `-`.

**Zapisuje:**

| plik | co zawiera |
|---|---|
| `results/figures/face_sizes_dev.png` | rozkład krótszego boku ramki twarzy, część deweloperska |
| `results/figures/face_sizes_test.png` | to samo dla części testowej |
| `<THESIS_FIGURES>/rozmiary_twarzy_<część>.png` | te same rysunki pod nazwami używanymi przez źródła `.tex`, tylko gdy `THESIS_FIGURES` jest ustawione |

In [ ]:
import importlib
import json
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation import figures, tables
from src.utils import experiments as exp

for module in (figures, tables, exp):
    importlib.reload(module)      # the kernel keeps a once-imported module in memory

MEASUREMENTS = ROOT / "results" / "measurements"
SPLITS = ("dev", "test")
SPLIT_NAMES = {"dev": "czesc deweloperska", "test": "czesc testowa"}
# The console tables above keep ASCII, as the console does everywhere in this
# repository; a figure label is not console output, so it carries the diacritics.
FIGURE_SPLIT_NAMES = {"dev": "część deweloperska",
                      "test": "część testowa"}
LABEL = exp.DATASET_NAMES

# The figure directory of the thesis repository, when there is one. None means
# the figure is written only into results/figures/. The copy there is called
# rozmiary_twarzy.png, which is the name the .tex sources use.
THESIS_FIGURES = None        # e.g. Path(r"C:\\Users\\PC\\...\\praca magisterska\\rysunki")


def newest(split):
    """The newest face_sizes measurement of one part, or None.

    The file name carries the date, so the newest one sorts last; the part is
    inside the payload, which is why the choice cannot be made on the name.
    """
    for path in sorted(MEASUREMENTS.glob("face_sizes_*.json"), reverse=True):
        record = json.loads(path.read_text(encoding="utf-8"))
        if record["data"].get("split") == split:
            return path, record["data"]
    return None


found = {split: newest(split) for split in SPLITS}
survey = {split: entry[1] for split, entry in found.items() if entry}

print("pomiar wczytany z:")
for split in SPLITS:
    entry = found[split]
    print(f"  {split:<6}{entry[0].name if entry else '- brak, uruchom face_size_survey.py'}")
if survey:
    any_split = next(iter(survey.values()))
    THRESHOLD = any_split["production_min_face_px"]
    CANDIDATES = sorted(int(k) for k in any_split["datasets"][
        next(iter(any_split["datasets"]))]["rejected_share"])
    print(f"\n  prog produkcyjny MIN_FACE_PX: {THRESHOLD} px")
    print(f"  podloga detekcji w pomiarze:  "
          f"{', '.join(str(s['floor_px']) for s in survey.values())} px")
    print(f"  progi kandydujace:            {CANDIDATES}")
else:
    THRESHOLD, CANDIDATES = None, []
    print("\nbrak pomiaru - dalsze bloki wypisza, czego brakuje")


## 1. Rozkład krótszego boku

Jeden wiersz na przedział rozmiaru, kolumny na zbiór i część: liczba twarzy, jej udział i średnia pewność detekcji w tym przedziale. Przedziały pochodzą z pomiaru (`EDGES` w `scripts/face_size_survey.py`), a próg `MIN_FACE_PX` leży dokładnie na jednej z ich granic, więc "poniżej progu" to suma pierwszych przedziałów, a nie ich część.

In [ ]:
for split in SPLITS:
    data = survey.get(split)
    if data is None:
        print(f"\n{SPLIT_NAMES[split]}: brak pomiaru - uruchom "
              f"scripts/face_size_survey.py --split {split}")
        continue

    order = [b["range"] for b in next(iter(data["datasets"].values()))["buckets"]]
    header = ["Krotszy bok [px]"]
    for dataset in data["datasets"]:
        header += [f"{LABEL[dataset]} twarzy", f"{LABEL[dataset]} udzial [%]",
                   f"{LABEL[dataset]} sr. pewnosc"]

    rows = []
    for position, name in enumerate(order):
        cells = [name]
        for dataset, summary in data["datasets"].items():
            bucket = summary["buckets"][position]
            cells += [bucket["count"], tables.number(bucket["share"], 2),
                      tables.number(bucket["mean_score"], 3)]
        rows.append(cells)
    rows.append(["razem"] + [c for summary in data["datasets"].values()
                             for c in (summary["faces"], "100,00",
                                       tables.number(None))])

    tables.show(f"Rozmiary twarzy, {SPLIT_NAMES[split]}", header, rows,
                note="Mediana krotszego boku: " + ", ".join(
                    f"{LABEL[d]} {tables.number(s['median_side'], 1)} px"
                    for d, s in data["datasets"].items())
                + f". Pomiar przy podlodze {data['floor_px']} px, wiec przedzialy "
                  f"ponizej progu {data['production_min_face_px']} px sa widoczne.")


## 2. Odsetek odrzucony przez próg

Dla każdego progu kandydującego: ile procent wykrytych twarzy odpada, ile zostaje i o ile trzeba je powiększyć do wejścia ArcFace'a. Wiersz progu produkcyjnego jest oznaczony.

In [ ]:
if not survey:
    print("brak pomiaru - uruchom scripts/face_size_survey.py")
else:
    header = ["Prog [px]", "Powiekszenie"]
    for split in SPLITS:
        data = survey.get(split)
        if data is None:
            continue
        for dataset in data["datasets"]:
            header.append(f"{LABEL[dataset]} / {split} odrzucone [%]")

    rows = []
    for threshold in CANDIDATES:
        key = str(threshold)
        mark = " <- MIN_FACE_PX" if threshold == THRESHOLD else ""
        upscale = next(s["upscale_to_crop"][key]
                       for data in survey.values()
                       for s in data["datasets"].values())
        cells = [f"{threshold}{mark}", f"{tables.number(upscale, 2)}x"]
        for split in SPLITS:
            data = survey.get(split)
            if data is None:
                continue
            for summary in data["datasets"].values():
                cells.append(tables.number(summary["rejected_share"][key], 2))
        rows.append(cells)

    tables.show("Twarze odrzucone przez prog", header, rows, align="lr" + "r" * (len(header) - 2),
                note="Powiekszenie to iloraz boku wycinka ArcFace'a i progu: przy 32 px "
                     "wycinek jest powiekszany, nie pomniejszany, wiec prog nizszy "
                     "oznacza wiecej twarzy o zmyslonych szczegolach, a nie wiecej "
                     "informacji.")

    missing_split = [s for s in SPLITS if s not in survey]
    if missing_split:
        print(f"\nbrak pomiaru dla: {', '.join(missing_split)}")


## 3. Rysunki

**Zapisuje:** `results/figures/face_sizes_dev.png` i `results/figures/face_sizes_test.png`. Gdy `THESIS_FIGURES` wskazuje katalog rysunków pracy, powstają tam kopie `rozmiary_twarzy_dev.png` i `rozmiary_twarzy_test.png`.

Osobny rysunek na część: dwa panele obok siebie wychodziły w kolumnie pracy za małe. Słupek na zbiór, oś pozioma po przedziałach rozmiaru. Pionowa linia stoi na progu `MIN_FACE_PX` - wszystko na lewo od niej bufor produkcyjny odrzuca. Skład jak w pracy (`usetex`); bez zainstalowanego LaTeX-a rysunek powstaje w krojach matplotliba i blok to wypisze.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

usetex = figures.style()
PCT = figures.percent_sign()
COLORS = {dataset: figures.OKABE_ITO[position]
          for position, dataset in enumerate(exp.SERIES)}

# One figure per part. Side by side the two panels came out too small to read
# at the width of a thesis page, and the two parts are never compared with each
# other on one axis anyway -- each is read against the same threshold line.
for split in SPLITS:
    data = survey.get(split)
    if data is None:
        print(f"{SPLIT_NAMES[split]}: brak pomiaru - uruchom "
              f"scripts/face_size_survey.py --split {split}")
        continue

    fig, axis = plt.subplots(figsize=(6.4, 4.0))
    names = [b["range"] for b in next(iter(data["datasets"].values()))["buckets"]]
    centres = np.arange(len(names))
    width = 0.8 / max(len(data["datasets"]), 1)
    for position, (dataset, summary) in enumerate(data["datasets"].items()):
        axis.bar(centres + (position - (len(data["datasets"]) - 1) / 2) * width,
                 [b["share"] for b in summary["buckets"]], width * 0.92,
                 color=COLORS.get(dataset, figures.OKABE_ITO[position]),
                 edgecolor="white", linewidth=0.6)
    edge = data["production_min_face_px"]
    # the threshold sits on a bucket boundary, so the line goes between two bars
    boundary = next((i for i, n in enumerate(names) if n.startswith(f"{edge}-")), None)
    if boundary is not None:
        axis.axvline(boundary - 0.5, color="0.35", ls="--", lw=1.2)
        axis.text(boundary - 0.45, axis.get_ylim()[1] * 0.95,
                  f"MIN\\_FACE\\_PX = {edge}" if usetex else f"MIN_FACE_PX = {edge}",
                  rotation=90, va="top", ha="left", fontsize=8.5, color="0.35")
    axis.set_xticks(centres)
    axis.set_xticklabels(names, rotation=45, ha="right", fontsize=9)
    axis.set_xlabel("krótszy bok ramki twarzy [px]")
    axis.set_ylabel(f"udział wykrytych twarzy [{PCT}]")

    handles = [Patch(facecolor=COLORS.get(d, figures.OKABE_ITO[i]), label=LABEL[d])
               for i, d in enumerate(data["datasets"])]
    fig.legend(handles=handles, loc="lower center", ncol=len(handles), frameon=False,
               bbox_to_anchor=(0.5, -0.04))
    fig.tight_layout(rect=(0, 0.06, 1, 1))
    print(f"\n{SPLIT_NAMES[split]}:")
    figures.save(fig, f"face_sizes_{split}", THESIS_FIGURES, f"rozmiary_twarzy_{split}")
    plt.show()